## LLM Fallback v2 — Clean Architecture

Uses the already-tuned BERT predictions as base and asks Groq only for entity pairs
that BERT missed on rare predicates. No re-decoding, no threshold issues.

In [13]:
import os, json, time, re
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from groq import Groq

# ════════════════════════════════════════════
# CONFIGURE HERE — only these lines change
# ════════════════════════════════════════════
GROQ_API_KEY = "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"

# File JSON già prodotto dall'inference unificata (threshold + tuning già applicati)
BERT_PREDICTIONS = Path("predictions/inference_bert_biomedbert_re_A5_hardneg.json")

# Predicati su cui attivare il fallback LLM
# Solo quelli con < 200 esempi gold — BERT non li impara bene
RARE_PREDICATES = {
    "strike", "change effect", "produced by", "compared to",
   # "change expression", "change abundance", "part of"
}

# Path dati
DEV_PATH      = Path("../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json")
GOLD_PATH     = Path("../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/json_format/train_gold.json")
FEWSHOT_PATH  = Path("../data/few_shot_examples.json")
OUTPUT_PATH   = Path("../predictions/inference_A5_llm_fallback.json")
# ════════════════════════════════════════════

client = Groq(api_key=GROQ_API_KEY)
print("Groq client initialized")
print(f"BERT predictions: {BERT_PREDICTIONS}")
print(f"Rare predicates:  {RARE_PREDICATES}")


Groq client initialized
BERT predictions: predictions\inference_bert_biomedbert_re_A5_hardneg.json
Rare predicates:  {'change abundance', 'change effect', 'part of', 'strike', 'produced by', 'change expression', 'compared to'}


## Labels and Legal Pairs

In [14]:
LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}
LEGAL_RELATIONS = [
    ("DDF","affect","DDF"),("microbiome","is linked to","DDF"),("DDF","target","human"),
    ("drug","change effect","DDF"),("DDF","is a","DDF"),("microbiome","located in","human"),
    ("chemical","influence","DDF"),("dietary supplement","influence","DDF"),("DDF","target","animal"),
    ("chemical","impact","microbiome"),("anatomical location","located in","animal"),
    ("microbiome","located in","animal"),("chemical","located in","anatomical location"),
    ("bacteria","part of","microbiome"),("DDF","strike","anatomical location"),
    ("drug","administered","animal"),("bacteria","influence","DDF"),("drug","impact","microbiome"),
    ("DDF","change abundance","microbiome"),("microbiome","located in","anatomical location"),
    ("microbiome","used by","biomedical technique"),("chemical","produced by","microbiome"),
    ("dietary supplement","impact","microbiome"),("bacteria","located in","animal"),
    ("animal","used by","biomedical technique"),("chemical","impact","bacteria"),
    ("chemical","located in","animal"),("food","impact","bacteria"),
    ("microbiome","compared to","microbiome"),("human","used by","biomedical technique"),
    ("bacteria","change expression","gene"),("chemical","located in","human"),
    ("drug","interact","chemical"),("food","administered","human"),
    ("DDF","change abundance","bacteria"),("chemical","interact","chemical"),
    ("chemical","part of","chemical"),("dietary supplement","impact","bacteria"),
    ("DDF","interact","chemical"),("food","impact","microbiome"),("food","influence","DDF"),
    ("bacteria","located in","human"),("dietary supplement","administered","human"),
    ("bacteria","interact","chemical"),("drug","change expression","gene"),
    ("drug","impact","bacteria"),("drug","administered","human"),
    ("anatomical location","located in","human"),("dietary supplement","change expression","gene"),
    ("chemical","change expression","gene"),("bacteria","interact","bacteria"),
    ("drug","interact","drug"),("microbiome","change expression","gene"),
    ("bacteria","interact","drug"),("food","change expression","gene"),
]

def norm_ent(l):
    if not l: return ""
    return "DDF" if str(l).strip().lower() == "ddf" else str(l).strip()

def norm_span(s):
    return re.sub(r"\s+", " ", str(s).strip())

legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    legal_pairs.setdefault((norm_ent(s), norm_ent(o)), set()).add(p)

print(f"Legal type pairs: {len(legal_pairs)}")


Legal type pairs: 52


## Load BERT Predictions, Dev Data and Gold

In [15]:
# Carica predizioni BERT già tuned
with open(BERT_PREDICTIONS) as f:
    bert_preds = json.load(f)

# Carica dev data per il testo di contesto
with open(DEV_PATH) as f:
    dev_data = json.load(f)

print(f"BERT predictions loaded: {len(bert_preds)} documents")
bert_total = sum(len(v.get("mention_level_relations",[])) for v in bert_preds.values())
print(f"Total BERT relations: {bert_total}")

# Conta predizioni BERT per predicato raro
bert_rare_counts = Counter()
for pmid, doc in bert_preds.items():
    for r in doc.get("mention_level_relations", []):
        if r["predicate"] in RARE_PREDICATES:
            bert_rare_counts[r["predicate"]] += 1

print(f"\nBERT predictions on rare predicates:")
for pred in sorted(RARE_PREDICATES):
    print(f"  {pred}: {bert_rare_counts.get(pred, 0)}")


BERT predictions loaded: 80 documents
Total BERT relations: 1136

BERT predictions on rare predicates:
  change abundance: 22
  change effect: 25
  change expression: 3
  compared to: 0
  part of: 35
  produced by: 13
  strike: 25


## Load or Build Few-Shot Examples

In [16]:
if FEWSHOT_PATH.exists():
    with open(FEWSHOT_PATH) as f:
        few_shot_examples = json.load(f)
    print(f"Few-shot examples loaded from {FEWSHOT_PATH}")
else:
    print("Building few-shot examples from gold training set...")
    few_shot_examples = defaultdict(list)

    with open(GOLD_PATH) as f:
        gold_data = json.load(f)

    for pmid, doc in gold_data.items():
        title    = doc["metadata"]["title"]
        abstract = doc["metadata"]["abstract"]
        full_text = f"{title} {abstract}"
        abstract_offset = len(title) + 1

        entities = {}
        for e in doc["entities"]:
            key = (e["text_span"], e["label"])
            offset = abstract_offset if e["location"] == "abstract" else 0
            entities[key] = {
                "text_span": e["text_span"], "label": e["label"],
                "start_idx": e["start_idx"] + offset,
                "end_idx":   e["end_idx"]   + offset,
            }

        for r in doc.get("mention_level_relations", []):
            pred = r["predicate"].strip()
            if len(few_shot_examples[pred]) >= 3:
                continue
            subj = entities.get((r["subject_text_span"], r["subject_label"]))
            obj  = entities.get((r["object_text_span"],  r["object_label"]))
            if not subj or not obj:
                continue
            left  = min(subj["start_idx"], obj["start_idx"])
            right = max(subj["end_idx"],   obj["end_idx"])
            context = full_text[max(0,left-150):min(len(full_text),right+150)].strip()
            few_shot_examples[pred].append({
                "context": context,
                "subject": r["subject_text_span"], "subject_label": r["subject_label"],
                "object":  r["object_text_span"],  "object_label":  r["object_label"],
            })

    FEWSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(FEWSHOT_PATH, "w") as f:
        json.dump(dict(few_shot_examples), f, ensure_ascii=False, indent=2)
    print(f"Saved to {FEWSHOT_PATH}")

print("\nFew-shot examples per predicate:")
for pred in sorted(RARE_PREDICATES):
    print(f"  {pred}: {len(few_shot_examples.get(pred, []))}")


Few-shot examples loaded from ..\data\few_shot_examples.json

Few-shot examples per predicate:
  change abundance: 3
  change effect: 3
  change expression: 3
  compared to: 3
  part of: 3
  produced by: 3
  strike: 3


## Identify Entity Pairs BERT Missed on Rare Predicates

For each document, find all legal entity pairs where:
1. The entity type pair could have a rare predicate
2. BERT did NOT predict any relation for that pair

In [17]:
def adjust_entity(e, abstract_offset):
    offset = abstract_offset if e["location"] == "abstract" else 0
    return {
        "text_span": norm_span(e["text_span"]),
        "label":     norm_ent(e["label"]),
        "start_idx": e["start_idx"] + offset,
        "end_idx":   e["end_idx"]   + offset,
        "location":  e["location"],
    }

# Coppie candidate per LLM: (pmid, subj, obj, legal_rare_preds, context)
llm_candidates = []

for pmid, article in dev_data.items():
    title    = article["metadata"]["title"]
    abstract = article["metadata"]["abstract"]
    full_text = f"{title} {abstract}"
    abstract_offset = len(title) + 1

    entities = [adjust_entity(e, abstract_offset) for e in article["entities"]]

    # Relazioni già predette da BERT per questo documento
    bert_doc_preds = bert_preds.get(str(pmid), {}).get("mention_level_relations", [])
    bert_predicted_pairs = set()
    for r in bert_doc_preds:
        k = (norm_span(r["subject_text_span"]), norm_ent(r["subject_label"]),
             norm_span(r["object_text_span"]),  norm_ent(r["object_label"]))
        bert_predicted_pairs.add(k)

    # Genera tutte le coppie legali con almeno un predicato raro
    for i, subj in enumerate(entities):
        for j, obj in enumerate(entities):
            if i == j:
                continue
            type_pair = (subj["label"], obj["label"])
            all_preds = legal_pairs.get(type_pair, set())
            rare_preds = legal_pairs.get((norm_ent(subj["label"]), norm_ent(obj["label"])), set()) \
             & RARE_PREDICATES
            if not rare_preds:
                continue

            k = (subj["text_span"], subj["label"], obj["text_span"], obj["label"])

            # Salta se BERT ha già predetto qualcosa per questa coppia
            if k in bert_predicted_pairs:
                continue
            # Aggiungi questo filtro nel loop candidati
            dist = abs(subj["start_idx"] - obj["start_idx"])
            if dist > 300:   # salta coppie troppo distanti
                continue
            # Estrai contesto
            left  = min(subj["start_idx"], obj["start_idx"])
            right = max(subj["end_idx"],   obj["end_idx"])
            win_start = max(0, left - 200)
            win_end   = min(len(full_text), right + 200)
            context = full_text[win_start:win_end].strip()

            llm_candidates.append({
                "pmid":    str(pmid),
                "subj_text":  subj["text_span"],
                "subj_label": subj["label"],
                "obj_text":   obj["text_span"],
                "obj_label":  obj["label"],
                "rare_preds": sorted(rare_preds),
                "all_preds":  sorted(all_preds),
                "context":    context,
            })

print(f"LLM candidates (pairs BERT missed on rare predicates): {len(llm_candidates)}")
candidate_counts = Counter()
for c in llm_candidates:
    for p in c["rare_preds"]:
        candidate_counts[p] += 1
print("By rare predicate:")
for pred, cnt in candidate_counts.most_common():
    print(f"  {pred}: {cnt}")


LLM candidates (pairs BERT missed on rare predicates): 12503
By rare predicate:
  part of: 4501
  change abundance: 3725
  strike: 1487
  compared to: 830
  produced by: 830
  change expression: 652
  change effect: 478


## Run LLM on Candidates

In [18]:
def build_prompt(context, subject, subject_label, obj, obj_label,
                  few_shot_examples, legal_predicates):
    lines = [
        "You are an expert biomedical relation extraction system.",
        "Given a biomedical text and two entities, classify their relation.",
        f"Subject entity: {subject} (type: {subject_label})",
        f"Object entity: {obj} (type: {obj_label})",
        f"Legal relations for this entity type pair: {', '.join(sorted(legal_predicates))}",
        "",
        "Output ONLY the relation label, nothing else.",
        "If no relation holds between these specific mentions, output: no relation",
        "",
        "--- EXAMPLES ---",
    ]

    n_shown = 0
    for pred in sorted(legal_predicates):
        if pred not in few_shot_examples:
            continue
        for ex in few_shot_examples[pred][:2]:
            lines += [
                f"Text: {ex['context'][:300]}",
                f"Subject: {ex['subject']} [{ex['subject_label']}]",
                f"Object: {ex['object']} [{ex['object_label']}]",
                f"Relation: {pred}", "",
            ]
            n_shown += 1
            if n_shown >= 6:
                break
        if n_shown >= 6:
            break

    lines += [
        "Text: The gut microbiota composition was analyzed in healthy controls.",
        "Subject: gut microbiota [microbiome]",
        "Object: Parkinson disease [DDF]",
        "Relation: no relation", "",
        "--- CLASSIFY ---",
        f"Text: {context[:400]}",
        f"Subject: {subject} [{subject_label}]",
        f"Object: {obj} [{obj_label}]",
        "Relation:",
    ]
    return "\n".join(lines)


def call_groq(prompt, model="llama-3.3-70b-versatile", max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=15,
                temperature=0.0,
            )
            return response.choices[0].message.content.strip().lower()
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return None
    return None


def parse_answer(answer, legal_predicates):
    if not answer:
        return "no relation"
    answer = answer.strip().lower().rstrip(".")
    if answer == "no relation":
        return "no relation"
    for pred in legal_predicates:
        if pred == answer:
            return pred
    for pred in legal_predicates:
        if pred in answer:
            return pred
    return "no relation"


print(f"Running LLM on {len(llm_candidates)} candidates...")
print("Estimated time: ~1 call/sec → ~{:.0f} min".format(len(llm_candidates)/60))
print()

llm_additions = defaultdict(list)  # pmid -> list of relation dicts
api_calls = 0
skipped   = 0
added     = 0

for cand in llm_candidates:
    pmid       = cand["pmid"]
    subj_text  = cand["subj_text"]
    subj_label = cand["subj_label"]
    obj_text   = cand["obj_text"]
    obj_label  = cand["obj_label"]
    rare_preds = cand["rare_preds"]
    all_preds  = cand["all_preds"]
    context    = cand["context"]

    prompt = build_prompt(
        context, subj_text, subj_label, obj_text, obj_label,
        few_shot_examples, set(rare_preds)
    )

    raw = call_groq(prompt)
    pred = parse_answer(raw, set(all_preds))
    api_calls += 1

    if pred != "no relation" and pred in rare_preds:
        llm_additions[pmid].append({
            "subject_text_span": subj_text,
            "subject_label":     subj_label,
            "predicate":         pred,
            "object_text_span":  obj_text,
            "object_label":      obj_label,
        })
        added += 1

    if api_calls % 50 == 0:
        print(f"  {api_calls}/{len(llm_candidates)} | added so far: {added}")

    time.sleep(0.15)

print(f"\nDone. API calls: {api_calls} | Added relations: {added} | Skipped: {skipped}")
added_counts = Counter()
for rels in llm_additions.values():
    for r in rels:
        added_counts[r["predicate"]] += 1
print("Added by predicate:")
for pred, cnt in added_counts.most_common():
    print(f"  {pred}: {cnt}")


Running LLM on 12503 candidates...
Estimated time: ~1 call/sec → ~208 min

  50/12503 | added so far: 41
  100/12503 | added so far: 67
  150/12503 | added so far: 78
  200/12503 | added so far: 88
  250/12503 | added so far: 88
  300/12503 | added so far: 89
  350/12503 | added so far: 89
  400/12503 | added so far: 90
  450/12503 | added so far: 90
  500/12503 | added so far: 90
  550/12503 | added so far: 90
  600/12503 | added so far: 91
  650/12503 | added so far: 91
  700/12503 | added so far: 92
  750/12503 | added so far: 93
  800/12503 | added so far: 93
  850/12503 | added so far: 93
  900/12503 | added so far: 94
  950/12503 | added so far: 95


KeyboardInterrupt: 

## Merge BERT + LLM and Evaluate

In [ ]:
def gold_tuple(r):
    return (norm_span(r["subject_text_span"]), norm_ent(r["subject_label"]),
            r["predicate"].strip(),
            norm_span(r["object_text_span"]),  norm_ent(r["object_label"]))

def build_gold_maps(dev_data):
    gold = {}
    for pmid, art in dev_data.items():
        s = set()
        for r in art.get("mention_level_relations", []):
            pred = r["predicate"].strip()
            if pred in LEGAL_RELATION_LABELS:
                s.add(gold_tuple(r))
        gold[str(pmid)] = s
    return gold

def pred_set_from_doc(doc):
    return {gold_tuple(r) for r in doc.get("mention_level_relations", [])
            if r["predicate"] in LEGAL_RELATION_LABELS}

def micro_scores(gold, pred):
    tp=fp=fn=0
    for pmid, g in gold.items():
        p = pred.get(pmid, set())
        tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    P=tp/(tp+fp) if (tp+fp) else 0.0
    R=tp/(tp+fn) if (tp+fn) else 0.0
    F1=2*P*R/(P+R) if (P+R) else 0.0
    return {"P":round(P,4),"R":round(R,4),"F1":round(F1,4),"TP":tp,"FP":fp,"FN":fn}

def macro_scores(gold, pred):
    preds_list = sorted({t[2] for s in gold.values() for t in s})
    vals = []
    for pr in preds_list:
        tp=fp=fn=0
        for pmid, g in gold.items():
            gp={t for t in g if t[2]==pr}
            pp={t for t in pred.get(pmid,set()) if t[2]==pr}
            tp+=len(gp&pp); fp+=len(pp-gp); fn+=len(gp-pp)
        P=tp/(tp+fp) if (tp+fp) else 0.0
        R=tp/(tp+fn) if (tp+fn) else 0.0
        vals.append((P,R,2*P*R/(P+R) if (P+R) else 0.0))
    return {
        "macro_P":  round(float(np.mean([x[0] for x in vals])),4),
        "macro_R":  round(float(np.mean([x[1] for x in vals])),4),
        "macro_F1": round(float(np.mean([x[2] for x in vals])),4),
    }

gold_by_doc = build_gold_maps(dev_data)

# BERT-only
bert_pred_by_doc = {str(pmid): pred_set_from_doc(doc)
                    for pmid, doc in bert_preds.items()}

# BERT + LLM merged
merged_pred_by_doc = {}
merged_out = {}
for pmid, doc in bert_preds.items():
    base_rels = list(doc.get("mention_level_relations", []))
    added_rels = llm_additions.get(str(pmid), [])
    all_rels = base_rels + added_rels
    # Deduplica
    seen = set()
    deduped = []
    for r in all_rels:
        k = gold_tuple(r)
        if k not in seen:
            seen.add(k)
            deduped.append(r)
    merged_pred_by_doc[str(pmid)] = {gold_tuple(r) for r in deduped
                                      if r["predicate"] in LEGAL_RELATION_LABELS}
    merged_out[str(pmid)] = {"mention_level_relations": deduped}

bert_mi = micro_scores(gold_by_doc, bert_pred_by_doc)
bert_ma = macro_scores(gold_by_doc, bert_pred_by_doc)
merged_mi = micro_scores(gold_by_doc, merged_pred_by_doc)
merged_ma = macro_scores(gold_by_doc, merged_pred_by_doc)

print("=== BERT only (A5) ===")
print(f"  Macro  P={bert_ma['macro_P']:.4f}  R={bert_ma['macro_R']:.4f}  F1={bert_ma['macro_F1']:.4f}")
print(f"  Micro  P={bert_mi['P']:.4f}  R={bert_mi['R']:.4f}  F1={bert_mi['F1']:.4f}")

print("\n=== BERT + LLM fallback ===")
print(f"  Macro  P={merged_ma['macro_P']:.4f}  R={merged_ma['macro_R']:.4f}  F1={merged_ma['macro_F1']:.4f}")
print(f"  Micro  P={merged_mi['P']:.4f}  R={merged_mi['R']:.4f}  F1={merged_mi['F1']:.4f}")

print(f"\n  Delta Macro F1: {merged_ma['macro_F1']-bert_ma['macro_F1']:+.4f}")
print(f"  Delta Micro F1: {merged_mi['F1']-bert_mi['F1']:+.4f}")
print(f"  LLM relations added: {added}")


## Save Merged Predictions

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(merged_out, f, ensure_ascii=False, indent=2)

total_rels = sum(len(v["mention_level_relations"]) for v in merged_out.values())
print(f"Saved: {OUTPUT_PATH}")
print(f"Documents: {len(merged_out)}")
print(f"Total relations: {total_rels}")
print(f"  BERT: {bert_total}")
print(f"  LLM additions: {added}")
